# 01 – Data Preprocessing

Loads the nationwide H3 grid dataset (parquet) produced by DuckDB,
translates Japanese park names in the `NAME` column to English slugs,
applies one-hot encoding to categorical geological and landform variables,
and saves the processed dataset for use in downstream modeling.

After this step all Japanese text is removed from the dataset;
subsequent notebooks work exclusively with English identifiers.

**Input**  : `data/interim/h3_jpn_res9_source_imputed.parquet`  
**Output** : `data/interim/h3_jpn_res9_processed.parquet`

Corresponds to *Section 2.2 – Data preprocessing* in the manuscript.

## Imports and configuration

In [ ]:
import pandas as pd

from config import DATA_DIR, PARK_NAME_MAP

## Load nationwide H3 dataset

In [ ]:
input_path = DATA_DIR / "h3_jpn_res9_source_imputed.parquet"
df_all = pd.read_parquet(input_path)

print(f"Loaded: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")
df_all.head()

## Translate park names to English slugs

The source GIS data uses Japanese park names in the `NAME` column.
These are translated to English slugs here so that all downstream
notebooks and output files are free of Japanese text.

In [ ]:
df_all["NAME"] = df_all["NAME"].map(PARK_NAME_MAP)

# Sanity check: rows outside any national park have NAME == NaN — expected.
# Rows inside a park that failed to map indicate a name mismatch.
park_rows = df_all[df_all["np_class"] == 1]
unmapped  = park_rows[park_rows["NAME"].isna()]
if not unmapped.empty:
    raise ValueError(f"{len(unmapped)} park rows could not be mapped to English slugs.")

print(f"Park name translation complete.")
print(f"Unique slugs ({df_all['NAME'].nunique()}): {sorted(df_all['NAME'].dropna().unique())}")

## One-hot encoding of categorical variables

Landform classification (`bichikei_en`) and geological group (`group_en`)
are one-hot encoded. The first category is dropped to avoid multicollinearity.

See *Section 2.2.1 – Data integration* in the manuscript.

In [ ]:
categorical_cols = ["bichikei_en", "group_en"]

df_all = pd.get_dummies(df_all, columns=categorical_cols, drop_first=True)

print(f"After encoding: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")

## Save processed dataset

In [ ]:
output_path = DATA_DIR / "h3_jpn_res9_processed.parquet"
df_all.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")